In [2]:
from huggingface_hub import login
login()

In [9]:
import torch
from selfcheckgpt.modeling_selfcheck import SelfCheckLLMPrompt

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# selfcheck_unigram = modeling_selfcheck.SelfCheckNgram(n=1)
# selfcheck_bigram = modeling_selfcheck.SelfCheckNgram(n=2)
# selfcheck_trigram = modeling_selfcheck.SelfCheckNgram(n=3)
# selfcheck_fourgram = modeling_selfcheck.SelfCheckNgram(n=4)
# selfcheck_fivegram = modeling_selfcheck.SelfCheckNgram(n=5)
# selfcheck_bertscore = modeling_selfcheck.SelfCheckBERTScore(rescale_with_baseline=True)
# selfcheck_nli = modeling_selfcheck.SelfCheckNLI(device=device)
# selfcheck_mqag = modeling_selfcheck.SelfCheckMQAG(device=device)
llm_model = "meta-llama/Llama-3.2-1B"
selfcheck_prompt = SelfCheckLLMPrompt(llm_model, device)

SelfCheck-LLMPrompt (meta-llama/Llama-3.2-1B) initialized to device cuda


In [6]:
import json

with open("data/dataset_v3.json", "r") as f:
    content = f.read()
dataset = json.loads(content)
print("The lenght of the dataset: {}".format(len(dataset)))
print(dataset[0].keys())

The lenght of the dataset: 238
dict_keys(['gpt3_text', 'wiki_bio_text', 'gpt3_sentences', 'annotation', 'wiki_bio_test_idx', 'gpt3_text_samples'])


In [7]:
import numpy as np

label_mapping = {
    'accurate': 0.0,
    'minor_inaccurate': 0.5,
    'major_inaccurate': 1.0,
}

human_label_detect_False   = {}
human_label_detect_True    = {}
human_label_detect_False_h = {}

for i_ in range(len(dataset)):
    dataset_i = dataset[i_]
    idx = dataset_i["wiki_bio_test_idx"]
    
    raw_label = np.array([label_mapping[x] for x in dataset_i['annotation']])
    human_label_detect_False[idx] = (raw_label > 0.499).astype(np.int32).tolist()
    human_label_detect_True[idx] = (raw_label < 0.499).astype(np.int32).tolist()
    average_score = np.mean(raw_label)
    if (average_score < 0.99):
        human_label_detect_False_h[idx] = (raw_label > 0.99).astype(np.int32).tolist()

print("Length of False:", len(human_label_detect_False))
print("Length of True:", len(human_label_detect_True)) 
print("Length of False_h:", len(human_label_detect_False_h))

Length of False: 238
Length of True: 238
Length of False_h: 206


In [10]:
from tqdm import tqdm

selfcheck_scores_prompt_llama = {}

for i in tqdm(range(len(dataset))):
    x = dataset[i]
    idx = dataset[i]['wiki_bio_test_idx'] 

    selfcheck_scores_prompt_llama[idx] = selfcheck_prompt.predict(
        sentences = x['gpt3_sentences'],           
        sampled_passages = x['gpt3_text_samples'],
        verbose=True
    )

  0%|          | 0/238 [00:00<?, ?it/s]


explanation: the not defined



explanation not defined


  0%|          | 1/238 [00:14<56:59, 14.43s/it]

  1%|          | 2/238 [00:26<51:16, 13.04s/it]

...

 96%|█████████▌| 228/238 [55:28<02:19, 13.92s/it]

 98%|█████████▊| 233/238 [56:42<01:15, 15.12s/it]

 98%|█████████▊| 234/238 [56:54<00:57, 14.43s/it]

100%|█████████▉| 237/238 [57:40<00:15, 15.81s/it]

100%|██████████| 238/238 [57:56<00:00, 14.61s/it]


In [11]:
selfcheck_scores_prompt_json = {}

for idx in selfcheck_scores_prompt_llama:
  scores = selfcheck_scores_prompt_llama[idx]
  selfcheck_scores_prompt_json[idx] = [score for score in scores]
  
with open("data/selfcheck_scores_prompt_llama.json", "w") as outfile:
    json.dump(selfcheck_scores_prompt_json, outfile)